In [ ]:
# take in  a small bit of ablations_data from sep24.3 for the 4 types and plot new metrics
# syntactical metrics and lexical metrics. so have to write out the new metrics and then use 
# the plotting code. write all inline. wrote out on gdoc

In [ ]:
# do literally 1 run of prop 0.5 with id 143 for 5 epochs and make sure wandb catches everything
# we want way more eval an logging turns per run.
# also want 

# make sure that everything is sensibly saved. logging files, csv for predictions, csv for metrics,
# run directories, and plotshould all go into one folder during a run.
# an example: sep24.3 has folders 'logs', 'plots', and 'runs'. in runs we have each run directory with the ablation_predictions and the calculated
# metrics from those predictions.  in the plot directory we have the plots auto created for each run. logs has slurm logs

# we prob for a small run want to print the data order in train, val, test
# also still want to have ablated test sets be pre generated.

 
# use cudf for the data stuff much later? 

In [ ]:
# do a small training run with 2 h200s on prop [0.0, 0.2, 0.5, 0.8, 1.0] and ids [1, 2]

In [1]:
import sys
sys.path.insert(0, "/n/home06/drooryck/codeswitching-llms")

import pandas as pd
from pathlib import Path

from july_aug_sept_exp.src.metrics import Metrics

lexicon_path = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
run_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p99.9_run5")
sample_n = 5000  # how many ablated predictions to sample

metrics = Metrics(lexicon_path)

preds = pd.read_csv(run_dir / "ablation_predictions.csv")
ablated = preds[preds["ablation"] != "none"].sample(n=sample_n, random_state=0)

scores = []
for text in ablated["prediction"]:
    tokens = metrics.tokenize(text)
    scores.append(metrics.lexical_mixture_score(tokens))

avg_score = sum(scores) / len(scores)
print(f"Average lexical score across {len(scores)} samples: {avg_score:.4f}")

/n/home06/drooryck/envs/codeswitching-py310/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Average lexical score across 5000 samples: 0.6896


In [9]:

df = pd.read_csv("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/results/nov11.0/runs/p00.10_run15/ablation_predictions.csv")
df[(df["ablation"] == "verb") & (df["language"] == "fr")].iloc[:10]

,language,input,gold,prediction,ablation
2,fr,le chien slaat le loup,le chien a frappé le loup,de gered hebben de gered geslagen,verb
6,fr,le chien slaat les loups,le chien a frappé les loups,de gered hebben de ezel geslagen,verb
10,fr,les chiens slaan le loup,les chiens ont frappé le loup,de varken heeft de gered geslagen,verb
14,fr,les chiens slaan les loups,les chiens ont frappé les loups,de varken heeft de ezel geslagen,verb
18,fr,le chien observeert le loup,le chien a observé le loup,de gered hebben de gered geobserveerd,verb
22,fr,le chien observeert les loups,le chien a observé les loups,de gered hebben de ezel geobserveerd,verb
26,fr,les chiens observeren le loup,les chiens ont observé le loup,de gered hebben de gered geobserveerd,verb
30,fr,les chiens observeren les loups,les chiens ont observé les loups,de varken heeft de ezel geobserveerd,verb
34,fr,le chien achtervolgt le loup,le chien a poursuivi le loup,de gered hebben de gered achtervolgd,verb
38,fr,le chien achtervolgt les loups,le chien a poursuivi les loups,de gered hebben de ezel achtervolgd,verb


In [ ]:
## do a big training run with the super debug method. 
# overnight nov11

# get the new plots on sep24.3 with lexical metrics and structure respecting.
# fix /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/plot_nov11_metrics.py

In [2]:
from pathlib import Path

from july_aug_sept_exp.src.plotting import BilingualPlotter

metrics_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3")
plots_dir = metrics_root / "temp_plots_nov11"

plotter = BilingualPlotter.create_plotter_from_run_metrics_dir(metrics_root, plots_subdir="temp_plots_nov11")
plotter.plot_nov11_structure_lexical()

print(f"Plots saved under {plots_dir}")

No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p95.0_run5
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p10.0_run2
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p85.0_run3
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p03.0_run3
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p92.5_run3
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p50.0_run1
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p40.0_run2
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p98.5_run2
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p40.0_run1
No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_e

ValueError: No valid results found in /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- configuration ---
results_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3")
ablations = ["verb", "object"]
plots_root = results_root / "temp_plots_nov11"
plots_root.mkdir(parents=True, exist_ok=True)

def collect_metrics(ablation: str) -> pd.DataFrame:
    rows = []
    required_keys = {
        "fr_follows_fr", "fr_follows_nl", "fr_follows_either", "fr_lexical_score",
        "nl_follows_fr", "nl_follows_nl", "nl_follows_either", "nl_lexical_score",
    }

    for run_dir in sorted(results_root.glob("p*_run*")):
        metrics_path = run_dir / f"ablation_{ablation}_metrics.json"
        if not metrics_path.exists():
            print(f"Skipping {run_dir.name}: {metrics_path.name} not found")
            continue

        metrics = json.loads(metrics_path.read_text())
        if not required_keys.issubset(metrics):
            print(f"Skipping {run_dir.name}: missing lexical metrics (run not recomputed yet)")
            continue

        prop_str, run_str = run_dir.name.split("_run")
        prop_pct = float(prop_str[1:])
        run_id = int(run_str)

        rows.append({
            "prop_pct": prop_pct,
            "prop_ratio": prop_pct / 100.0,
            "run_id": run_id,
            "fr_follows_fr": metrics["fr_follows_fr"],
            "fr_follows_nl": metrics["fr_follows_nl"],
            "fr_follows_either": metrics["fr_follows_either"],
            "fr_lexical_score": metrics["fr_lexical_score"],
            "nl_follows_fr": metrics["nl_follows_fr"],
            "nl_follows_nl": metrics["nl_follows_nl"],
            "nl_follows_either": metrics["nl_follows_either"],
            "nl_lexical_score": metrics["nl_lexical_score"],
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No usable metrics found for ablation='{ablation}'—rerun the recompute step first.")
    return df

def aggregate_language(df_lang: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = {
        f"{prefix}_follows_fr": "syntax_fr",
        f"{prefix}_follows_nl": "syntax_nl",
        f"{prefix}_follows_either": "syntax_either",
        f"{prefix}_lexical_score": "lexical",
    }
    agg = (
        df_lang.groupby("prop_ratio")[list(cols.keys())]
        .agg(["mean", "std", "count"])
        .rename(columns=cols, level=0)
    )
    agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
    for metric in cols.values():
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        count_col = f"{metric}_count"
        sem_col = f"{metric}_sem"
        agg[sem_col] = agg[std_col] / np.sqrt(agg[count_col].clip(lower=1))
    return agg.reset_index()

colors = {
    "syntax_fr": "#1f77b4",
    "syntax_nl": "#ff7f0e",
    "syntax_either": "#2ca02c",
    "lexical": "#d62728",
}

for ablation in ablations:
    df = collect_metrics(ablation)
    fr_agg = aggregate_language(df, "fr")
    nl_agg = aggregate_language(df, "nl")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    axes_info = [
        (axes[0], fr_agg, "French inputs"),
        (axes[1], nl_agg, "Dutch inputs"),
    ]

    for ax, agg_df, title in axes_info:
        ax.set_title(title)
        ax.set_xlabel("Training French proportion")
        ax.set_ylabel("Mean score across runs")
        for metric, label in [
            ("syntax_fr", "French structure"),
            ("syntax_nl", "Dutch structure"),
            ("syntax_either", "Either structure"),
            ("lexical", "Lexical FR share"),
        ]:
            ax.errorbar(
                agg_df["prop_ratio"],
                agg_df[f"{metric}_mean"],
                yerr=agg_df[f"{metric}_sem"],
                marker="o",
                linewidth=2,
                capsize=4,
                color=colors[metric],
                label=label,
            )
        ax.set_ylim(0, 1)
        ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.6)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4)
    fig.tight_layout(rect=[0, 0, 1, 0.92])

    output_path = plots_root / f"nov11_{ablation}_mean_sem.png"
    fig.savefig(output_path, dpi=200)
    plt.close(fig)
    print(f"Saved plot for ablation='{ablation}' to {output_path}")

Skipping p99.5_run5: missing lexical metrics (run not recomputed yet)
Saved plot for ablation='verb' to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/temp_plots_nov11/nov11_verb_mean_sem.png
Skipping p99.5_run4: missing lexical metrics (run not recomputed yet)
Skipping p99.5_run5: missing lexical metrics (run not recomputed yet)
Saved plot for ablation='object' to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/temp_plots_nov11/nov11_object_mean_sem.png
